# 04 — SHAP Explainability Analysis (PRD Phase 4)

TreeSHAP attribution for the fraud ensemble's **XGBoost component** — global
importance, the per-transaction SHAP distribution, worked single-transaction
explanations for a true positive / false negative / false positive, and the
`TransactionAmt` dependence.

**Scope (ADR-001 §3.4).** SHAP here is exact for the XGBoost sub-model, which
carries **0.692** of the calibrated blend. It is *not* an explanation of the
ensemble decision — LightGBM (0.174) and the TFT (0.134) are not represented.
KernelSHAP over the blend was rejected on latency grounds in the ADR. Every
contribution is in **raw-margin (log-odds) space**: `base_value + Σcontributions`
reconstructs the booster's margin for the row.

Reuses `src/explainability/shap_explainer.py` (`FraudExplainer`) — the same
object the serving path builds at startup — so this notebook and production
cannot disagree about the attribution.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml

with open(PROJECT_ROOT / 'config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'
RNG = np.random.default_rng(42)

## 1. Load the test split and the frozen XGBoost model

In [2]:
processed_dir = PROJECT_ROOT / config['data']['processed_dir']

X_test = pd.read_parquet(processed_dir / 'test_features.parquet')
y_test = pd.read_parquet(processed_dir / 'test_labels.parquet').squeeze()

print(f'Test set: {len(X_test):,} rows, fraud rate: {y_test.mean():.4f}')

Test set: 118,109 rows, fraud rate: 0.0344


In [3]:
from src.training.train_xgb import XGBTrainer

xgb_trainer = XGBTrainer.load(str(PROJECT_ROOT / 'models/xgb_model.pkl'))
print(f'XGBoost: {len(xgb_trainer.feature_names)} features, frozen threshold '
      f'{xgb_trainer.threshold:.6g} (raw-probability space)')

import json as _json
_spec = _json.loads((PROJECT_ROOT / 'models/ensemble.json').read_text())
XGB_WEIGHT = _spec['modes'][_spec['default_mode']]['weights']['xgb']
print(f'XGBoost blend weight: {XGB_WEIGHT:.3f}')

2026-09-08 11:31:52,377 [INFO] src.training.train_xgb: Model loaded from c:\Users\Admin\Desktop\projects\fraud-xai trial\models\xgb_model.ubj


XGBoost: 171 features, frozen threshold 0.000481583 (raw-probability space)
XGBoost blend weight: 0.692


## 2. Build the explainer

`FraudExplainer` wraps `shap.TreeExplainer` on the booster and calibrates the
additive base value once (see the module docstring for the `shap==0.44`
`expected_value` drift it works around).


In [4]:
from src.explainability.shap_explainer import FraudExplainer

explainer = FraudExplainer(
    xgb_trainer.model,
    feature_names=xgb_trainer.feature_names,
    top_k=10,
    explained_weight=XGB_WEIGHT,
)
print(f'base_value (raw margin): {explainer.base_value:.6f}')

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
[11:31:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0b3782d1791676daf-1\xgboost\xgboost-ci-windows\src\c_api\c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
2026-09-08 11:32:00,532 [INFO] src.explainability.shap_explainer: FraudExplainer ready: 171 features, base_value=0.858635 (raw margin), explains model=xgb weight=0.6920000000000001


base_value (raw margin): 0.858635


## 3. SHAP over a test sample

TreeSHAP is exact and fast, but 118k rows renders to a smear — a 4,000-row
random sample is plenty for stable global statistics.


In [5]:
SAMPLE_N = 4000
sample_idx = RNG.choice(len(X_test), size=min(SAMPLE_N, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx].reset_index(drop=True)
y_sample = y_test.iloc[sample_idx].reset_index(drop=True)

shap_values = explainer.explain_batch(X_sample)   # (n_rows, n_features), trained order
aligned = X_sample.loc[:, xgb_trainer.feature_names]
print('SHAP matrix:', shap_values.shape)

margin = xgb_trainer.model.predict(aligned, output_margin=True)
recon = explainer.base_value + shap_values.sum(axis=1)
print(f'max |base+Σshap - margin| = {np.abs(recon - margin).max():.2e}')

SHAP matrix: (4000, 171)
max |base+Σshap - margin| = 2.62e-05


## 4. Global feature importance — mean(|SHAP|)

In [6]:
importance = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=xgb_trainer.feature_names)
    .sort_values(ascending=False)
)
TOP_K = 20
top = importance.head(TOP_K)

fig, ax = plt.subplots(figsize=(8, 0.42 * TOP_K))
ax.barh(top.index[::-1], top.to_numpy()[::-1], color='#c0392b')
ax.set_xlabel('mean(|SHAP value|)  —  average impact on XGBoost log-odds')
ax.set_title(f'Global feature importance (XGBoost component, n={len(X_sample):,})')
ax.grid(axis='x', alpha=0.3)
fig.savefig(FIGURES_DIR / 'shap_global_importance.png')
plt.show()

top.round(5)

FigureCanvasAgg is non-interactive, and thus cannot be shown


C13                     0.40104
card1_target_enc        0.35225
C1                      0.34845
C14                     0.30206
C5                      0.28807
card1                   0.27440
card_hash_freq          0.26194
TransactionAmt          0.23466
card2                   0.22248
addr1                   0.22199
pca_v_1                 0.21547
card6                   0.20977
tx_sum_per_card         0.20508
mean_amount_per_card    0.20108
pca_v_7                 0.18616
tx_count_per_card       0.18089
M5                      0.17416
card2_target_enc        0.17338
std_amount_per_card     0.17058
max_amount_per_card     0.17040
dtype: float64

## 5. Per-transaction SHAP distribution (beeswarm)

Each point is one transaction's SHAP value for that feature; colour is the
feature value (blue = low, red = high). Spread away from zero means the feature
moved that transaction's score.


In [7]:
order = list(top.index)
idx = [aligned.columns.get_loc(f) for f in order]
shap_sub = shap_values[:, idx]
feat_sub = aligned.iloc[:, idx].to_numpy()

fig, ax = plt.subplots(figsize=(8, 0.42 * len(order)))
jit = np.random.default_rng(0)
for row, feat in enumerate(order[::-1]):
    col = len(order) - 1 - row
    vals = shap_sub[:, col]
    craw = feat_sub[:, col].astype(float)
    spread = np.ptp(craw)
    colour = (craw - craw.min()) / spread if spread else np.zeros_like(craw)
    ax.scatter(vals, np.full(len(vals), row) + jit.uniform(-0.18, 0.18, len(vals)),
               c=colour, cmap='coolwarm', s=8, alpha=0.6, linewidths=0)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order[::-1])
ax.axvline(0, color='#333', linewidth=0.8)
ax.set_xlabel('SHAP value (impact on XGBoost log-odds)')
ax.set_title('Per-transaction SHAP distribution — colour = feature value (low→high)')
ax.grid(axis='x', alpha=0.3)
fig.savefig(FIGURES_DIR / 'shap_beeswarm.png')
plt.show()

FigureCanvasAgg is non-interactive, and thus cannot be shown


## 6. Worked single-transaction explanations — TP / FN / FP

**Note on case selection.** Cases below are picked at the **XGBoost sub-model's
own operating point** (its raw-probability threshold), which is *not* the
ensemble's served decision boundary (the frozen `0.006123` over the calibrated
3-model blend, ADR-001 §3.4). A transaction the notebook labels a "miss" may
still be flagged by the deployed ensemble, and vice versa — this section
explains XGBoost's contribution, not the production verdict.

The regulator-facing case: *why did the model decide this about this
transaction?* We pick one true positive, one false negative, and one false
positive at the frozen operating point, and render each as a waterfall from the
base value to the booster margin.


In [8]:
p_raw = xgb_trainer.predict_proba(X_sample)
pred = (p_raw >= xgb_trainer.threshold).astype(int)

def _pick(mask):
    hits = np.where(mask)[0]
    return int(hits[0]) if len(hits) else None

cases = {
    'True positive (fraud, flagged)':  _pick((y_sample.to_numpy() == 1) & (pred == 1)),
    'False negative (fraud, missed)':  _pick((y_sample.to_numpy() == 1) & (pred == 0)),
    'False positive (legit, flagged)': _pick((y_sample.to_numpy() == 0) & (pred == 1)),
}
print({k: v for k, v in cases.items()})

{'True positive (fraud, flagged)': 3, 'False negative (fraud, missed)': 621, 'False positive (legit, flagged)': 0}


In [9]:
def waterfall(row_pos, title, path):
    contrib = pd.Series(shap_values[row_pos], index=xgb_trainer.feature_names)
    order_ = contrib.abs().sort_values(ascending=False).head(12).index
    c = contrib[order_][::-1]

    base = explainer.base_value
    running = base + (contrib.sum() - c.sum())  # lump the un-shown tail into the start
    lefts, colors = [], []
    for v in c:
        lefts.append(running if v >= 0 else running + v)
        colors.append('#c0392b' if v >= 0 else '#2c7fb8')
        running += v

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(c)), c.to_numpy(), left=lefts, color=colors)
    ax.set_yticks(range(len(c)))
    ax.set_yticklabels(c.index)
    ax.axvline(base, color='#888', linestyle='--', linewidth=1, label=f'base {base:.2f}')
    final_margin = base + contrib.sum()
    ax.axvline(final_margin, color='#111', linewidth=1.2,
               label=f'margin {final_margin:.2f}  (p={1/(1+np.exp(-final_margin)):.3f})')
    ax.set_xlabel('XGBoost log-odds (margin)')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(axis='x', alpha=0.3)
    fig.savefig(path)
    plt.show()

for name, pos in cases.items():
    if pos is None:
        print(f'(no {name} in the sample)')
        continue
    slug = name.split('(')[0].strip().lower().replace(' ', '_')
    waterfall(pos, f'{name} — transaction #{pos}', FIGURES_DIR / f'shap_waterfall_{slug}.png')

FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown
FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. `TransactionAmt` dependence

How the model's `TransactionAmt` contribution varies with the amount itself —
the single feature a fraud reviewer will ask about first.


In [10]:
amt_col = 'TransactionAmt'
if amt_col in aligned.columns:
    j = aligned.columns.get_loc(amt_col)
    amt = aligned.iloc[:, j].to_numpy()
    amt_shap = shap_values[:, j]

    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(amt, amt_shap, c=y_sample.to_numpy(), cmap='coolwarm',
                    s=10, alpha=0.5, linewidths=0)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(mtick.FormatStrFormatter('%d'))
    ax.axhline(0, color='#333', linewidth=0.8)
    ax.set_xlabel('TransactionAmt (log scale)')
    ax.set_ylabel('SHAP value for TransactionAmt (log-odds)')
    ax.set_title('TransactionAmt dependence — colour = actual label (blue=legit, red=fraud)')
    ax.grid(alpha=0.3)
    fig.colorbar(sc, ax=ax, label='isFraud')
    fig.savefig(FIGURES_DIR / 'shap_transactionamt_dependence.png')
    plt.show()
else:
    print(f'{amt_col!r} not in the engineered feature set')

FigureCanvasAgg is non-interactive, and thus cannot be shown


## 8. Regulatory note

**What this explanation is.** For every scored transaction the API returns a
signed contribution per feature (`PredictionResponse.explanation`), computed with
TreeSHAP. For a tree model TreeSHAP is *exact* — no sampling, no background
dataset — and the contributions sum to the model's output:
`base_value + Σ contributions = XGBoost margin` (verified to ~1e-6 in §3). The
values are in log-odds units; a positive contribution pushed the transaction
toward "fraud".

**What it covers.** The explanation is of the **XGBoost sub-model only**, which
is 0.692 of the deployed blend (`explained_model` and `explained_weight` are on
every response). It is a faithful account of the largest single component of the
decision, not of the ensemble output. This scope was chosen in ADR-001 §3.4:
an exact, sub-second explanation of the dominant component was preferred over an
approximate (KernelSHAP) explanation of the whole blend that would not meet the
serving latency budget.

**Adverse-action use.** The top positive contributions for a declined
transaction are the model's stated reasons. Because the contribution set is over
*engineered* features (e.g. `card1` frequency, per-card amount z-score), mapping
each to a customer-facing reason code is a required downstream step before these
are shown to an applicant — the raw feature names are not adverse-action
language.

**Stability.** The base value and the TreeSHAP algorithm are fixed by the frozen
model artifact; re-running this notebook on the same artifact reproduces the
attribution. A model retrain changes it, and `monitoring/shap_dashboard.html`
should be regenerated at that point (`scripts/generate_shap_dashboard.py`).

**Coverage monitoring.** `serving_metrics.explanation_failures` on `/health`
counts requests whose SHAP step raised; the score is always returned regardless,
so a non-zero value means degraded explanation coverage, never a degraded
decision.
